# Basic Modeling for Seizure Detection

## Loading in data, filtering for patients with positive samples

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

: 

In [2]:
import os

os.chdir("./drive/MyDrive/MIT/6.S985/ds005873")

In [3]:
list_windows = os.listdir('./processed_windows')

data_windows = [l for l in list_windows if l.endswith('.h5')]

In [4]:
with open(r'/content/drive/MyDrive/MIT/6.S985/ds005873/all_eeg_paths.csv', 'r') as f:
  all_eeg_paths = f.read().split(',')

with open(r'/content/drive/MyDrive/MIT/6.S985/ds005873/all_ecg_paths.csv', 'r') as f:
  all_ecg_paths = f.read().split(',')

In [5]:
print(len(data_windows))

2804


In [6]:
import h5py
import mne
import pandas as pd
from tqdm import tqdm


data_dict = {}

for path in tqdm(data_windows):

  output_prefix = "/content/drive/MyDrive/MIT/6.S985/ds005873/processed_windows/" + path.split('.')[0]
  h5_path = output_prefix + ".h5"
  meta_path = output_prefix + "_metadata.parquet"
  info_path = output_prefix + "_info.json"

  # h5py_file = h5py.File(h5_path, 'r')

  # Example: load ECG windows
  # with h5py.File(h5_path, 'r') as f:
  #   if "ecg_windows" in f:
  #     ecg = f["ecg_windows"][:]      # ← this loads as a NumPy array
  #     # print("ECG shape:", ecg.shape)

  #   # Example: load EEG windows
  #   if "eeg_windows" in f:
  #     eeg = f["eeg_windows"][:]
  #     # print("EEG shape:", eeg.shape)

  with open(meta_path, 'rb') as f:
    meta = pd.read_parquet(f)

  if meta['binary_label'].sum() > 0:
    data_dict[meta['run_id'].iloc[0]] = {
        'num_pos': meta['binary_label'].sum(),
        'size': len(meta['binary_label']),
        'patient_id': meta['patient_id'].iloc[0],
        'prefix': output_prefix
    }


ModuleNotFoundError: No module named 'mne'

In [15]:
import pandas as pd

df = pd.DataFrame(data_dict).T

In [22]:
df['num_pos'].sum()

np.int64(11554)

In [23]:
df.to_csv('./patients_w_pos_samples.csv')

In [7]:
import pandas as pd

df = pd.read_csv("./patients_w_pos_samples.csv")

In [8]:
from sklearn.model_selection import train_test_split

all_patients = df['patient_id'].unique()

train_patients, test_patients = train_test_split(all_patients, test_size=0.2, random_state=42)
train_patients, val_patients = train_test_split(train_patients, test_size=0.2, random_state=42)

In [9]:
df['size'].sum()

np.int64(2082570)

In [10]:
train_ids = df[df['patient_id'].isin(train_patients)]
val_ids = df[df['patient_id'].isin(val_patients)]
test_ids = df[df['patient_id'].isin(test_patients)]

In [11]:
train_ids['size']

,size
4,305
5,450
6,598
7,1187
8,308
...,...
373,16254
374,4755
375,15040
376,706


In [12]:
import os
import h5py
import pyarrow.parquet as pq
import numpy as np
import torch
from tqdm import tqdm

INPUT_DIR = "/content/drive/MyDrive/MIT/6.S985/ds005873/processed_windows"
OUTPUT_DIR = "/content/drive/MyDrive/MIT/6.S985/ds005873/consolidated"
POS_NEG_RATIO = 4  # 1 positive : 4 negative

def process_dataset_by_split(bases, split):
    eeg_list = []
    ecg_list = []

    # store all labels
    binary_label_list = []
    lateralization_list = []
    label_list = []
    localization_list = []
    vigilance_list = []
    duration_list = []

    for base in tqdm(bases):
        h5_path = os.path.join(INPUT_DIR, base + ".h5")
        meta_path = os.path.join(INPUT_DIR, base + "_metadata.parquet")

        # load metadata
        meta_df = pq.read_table(meta_path).to_pandas()

        labels = meta_df["binary_label"].to_numpy()
        lateralization = meta_df["lateralization"].to_numpy()
        other_label = meta_df["label"].to_numpy()  # assuming this is the "label" column
        localization = meta_df["localization"].to_numpy()
        vigilance = meta_df["vigilance"].to_numpy()
        duration = meta_df["seizure_duration_sec"].to_numpy(dtype=np.float32)

        # indices of positives and negatives
        pos_idx = np.where(labels == 1)[0]
        neg_idx = np.where(labels == 0)[0]

        # sample negatives according to ratio
        n_neg_sample = min(len(neg_idx), len(pos_idx) * POS_NEG_RATIO)
        neg_sample_idx = np.random.choice(neg_idx, size=n_neg_sample, replace=False)

        # combine
        selected_idx = np.concatenate([pos_idx, neg_sample_idx])
        np.random.shuffle(selected_idx)

        with h5py.File(h5_path, "r") as f:
            if "eeg_windows" in f:
                eeg = np.take(f["eeg_windows"], selected_idx, axis=0).astype(np.float32)
                eeg_list.append(eeg)

            if "ecg_windows" in f:
                ecg = np.take(f["ecg_windows"], selected_idx, axis=0).astype(np.float32)
                ecg_list.append(ecg)

        # append sampled labels
        binary_label_list.append(labels[selected_idx])
        lateralization_list.append(lateralization[selected_idx].astype(object))
        label_list.append(other_label[selected_idx].astype(object))
        localization_list.append(localization[selected_idx].astype(object))
        vigilance_list.append(vigilance[selected_idx].astype(object))
        duration_list.append(duration[selected_idx])

    # concatenate all files
    eeg_np = np.concatenate(eeg_list, axis=0)
    ecg_np = np.concatenate(ecg_list, axis=0)

    binary_labels_np = np.concatenate(binary_label_list, axis=0)
    lateralization_np = np.concatenate(lateralization_list, axis=0)
    other_label_np = np.concatenate(label_list, axis=0)
    localization_np = np.concatenate(localization_list, axis=0)
    vigilance_np = np.concatenate(vigilance_list, axis=0)
    duration_np = np.concatenate(duration_list, axis=0)

    print("Final shapes:")
    print("EEG:", eeg_np.shape, "ECG:", ecg_np.shape, "Binary labels:", binary_labels_np.shape)

    # create output directory
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # save all arrays as compressed npz
    output_path = os.path.join(OUTPUT_DIR, f"{split}_data.npz")
    np.savez_compressed(
        output_path,
        eeg=eeg_np,
        ecg=ecg_np,
        binary_label=binary_labels_np,
        lateralization=lateralization_np,
        label=other_label_np,
        localization=localization_np,
        vigilance=vigilance_np,
        seizure_duration_sec=duration_np
    )

    print(f"Saved to {output_path}")

In [13]:
# process_dataset_by_split(train_ids['prefix'], 'train')
process_dataset_by_split(val_ids['prefix'], 'val')
process_dataset_by_split(test_ids['prefix'], 'test')

100%|██████████| 53/53 [03:39<00:00,  4.14s/it]


Final shapes:
EEG: (9070, 2, 2560) ECG: (9070, 1, 2560) Binary labels: (9070,)
Saved to /content/drive/MyDrive/MIT/6.S985/ds005873/consolidated/val_data.npz


100%|██████████| 83/83 [05:20<00:00,  3.86s/it]


Final shapes:
EEG: (11360, 2, 2560) ECG: (11360, 1, 2560) Binary labels: (11360,)
Saved to /content/drive/MyDrive/MIT/6.S985/ds005873/consolidated/test_data.npz


## Modeling starts here

In [37]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

class SeizureDataset(Dataset):

  def __init__(self, data_path, eeg_transform=None, ecg_transform=None):
    data = np.load(data_path, allow_pickle=True)

    self.eeg = data['eeg']
    self.ecg = data['ecg']

    self.binary_label=data['binary_label']
    self.lateralization=data['lateralization']
    self.label=data['label']
    self.localization=data['localization']
    self.vigilance=data['vigilance']
    self.seizure_duration_sec=data['seizure_duration_sec']

    self.eeg_transform = eeg_transform
    self.ecg_transform = ecg_transform

  def __len__(self):
    return len(self.eeg)

  def __getitem__(self, idx):
    eeg = self.eeg[idx]
    ecg = self.ecg[idx]
    label = self.binary_label[idx]

    if self.eeg_transform:
      eeg = self.eeg_transform(eeg)

    if self.ecg_transform:
      ecg = self.ecg_transform(ecg)

    return ecg, eeg, label

In [38]:
trainset = SeizureDataset('./consolidated/train_data.npz')
valset = SeizureDataset('./consolidated/val_data.npz')
testset = SeizureDataset('./consolidated/test_data.npz')

In [39]:
trainloader = DataLoader(trainset, batch_size=8, shuffle=True)
valloader = DataLoader(valset, batch_size=8, shuffle=False)
testloader = DataLoader(testset, batch_size=8, shuffle=False)

In [61]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TwoModalityMLP(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()

        # Modality 1 (vector)
        self.m1 = nn.Linear(2560, hidden)

        # Modality 2 (2 × 2560 signal)
        self.conv = nn.Sequential(
            nn.Conv1d(2, 16, kernel_size=7, stride=2, padding=3),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)   # output → (B, 32, 1)
        )
        self.m2 = nn.Linear(32, hidden)

        # Fusion
        self.fuse = nn.Linear(2 * hidden, 1)

    def forward(self, x1, x2):
        # x1: (B, 1, dim1)

        # x2: (B, 2, 2560)
        h1 = F.relu(self.m1(x1)).squeeze()

        feat = self.conv(x2).squeeze(-1)  # (B, 32)
        h2 = F.relu(self.m2(feat))

        h = torch.cat([h1, h2], dim=1)
        return self.fuse(h)

In [62]:
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

def train_model(model, trainloader, valloader, epochs, lr=1e-4, device='cuda'):
    model = model.to(device)
    optim = torch.optim.Adam(model.parameters(), lr=lr)
    pos_w = torch.tensor([4.0], device=device)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    for ep in range(1, epochs + 1):
        model.train()
        pbar = tqdm(trainloader, desc=f"Epoch {ep}")

        for m1, m2, y in pbar:
            m1, m2, y = m1.to(device), m2.to(device), y.to(device).float()
            pred = model(m1, m2).squeeze()
            loss = bce(pred, y)

            optim.zero_grad()
            loss.backward()
            optim.step()
            # pbar.set_postfix(loss=loss.item())

        # ---- Validation AUC ----
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for m1, m2, y in valloader:
                m1, m2 = m1.to(device), bam2.to(device)
                pred = model(m1, m2).squeeze().cpu()
                preds.append(pred)
                trues.append(y)

        preds = torch.cat(preds).numpy()
        trues = torch.cat(trues).numpy()
        auc = roc_auc_score(trues, preds)

        print(f"Epoch {ep} | Val AUC: {auc:.4f}")

In [65]:
model = TwoModalityMLP()

train_model(model, trainloader, valloader, epochs=10, device='cpu')

Epoch 1: 100%|██████████| 4634/4634 [01:09<00:00, 66.25it/s]


Epoch 1 | Val AUC: 0.5849


Epoch 2: 100%|██████████| 4634/4634 [01:03<00:00, 73.55it/s]


Epoch 2 | Val AUC: 0.6006


Epoch 3: 100%|██████████| 4634/4634 [01:01<00:00, 75.12it/s]


...

Epoch 4 | Val AUC: 0.6353


Epoch 5: 100%|██████████| 4634/4634 [01:01<00:00, 75.13it/s]


Epoch 5 | Val AUC: 0.6430


Epoch 6:   2%|▏         | 94/4634 [00:01<01:21, 55.53it/s]


KeyboardInterrupt: 

In [66]:
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)
import torch

def test_model(model, testloader, device='cuda'):
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for m1, m2, y in testloader:
            m1, m2 = m1.to(device), m2.to(device)
            logits = model(m1, m2).squeeze()
            probs = torch.sigmoid(logits).cpu()

            all_probs.append(probs)
            all_labels.append(y)

    probs = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()

    # ---- Metrics ----
    auc = roc_auc_score(labels, probs)
    preds = (probs >= 0.5).astype(int)

    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec  = recall_score(labels, preds, zero_division=0)
    f1   = f1_score(labels, preds, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    specificity = tn / (tn + fp + 1e-9)
    sensitivity = tp / (tp + fn + 1e-9)

    # ---- Print summary ----
    print("==== Test Results ====")
    print(f"AUC:          {auc:.4f}")
    print(f"Accuracy:     {acc:.4f}")
    print(f"Precision:    {prec:.4f}")
    print(f"Recall:       {rec:.4f}")
    print(f"F1 Score:     {f1:.4f}")
    print(f"Sensitivity:  {sensitivity:.4f}")
    print(f"Specificity:  {specificity:.4f}")
    print(f"TP/FP/FN/TN:  {tp}/{fp}/{fn}/{tn}")

    return {
        "auc": auc,
        "acc": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn,
    }

In [67]:
test_model(model, testloader, device='cpu')

==== Test Results ====
AUC:          0.6395
Accuracy:     0.7210
Precision:    0.3353
Recall:       0.4018
F1 Score:     0.3656
Sensitivity:  0.4018
Specificity:  0.8008
TP/FP/FN/TN:  913/1810/1359/7278


{'auc': np.float64(0.6395074548855633),
 'acc': 0.7210387323943662,
 'precision': 0.33529195739992657,
 'recall': 0.40184859154929575,
 'f1': 0.3655655655655656,
 'sensitivity': np.float64(0.4018485915491189),
 'specificity': np.float64(0.8008362676055456),
 'tp': np.int64(913),
 'fp': np.int64(1810),
 'fn': np.int64(1359),
 'tn': np.int64(7278)}